# **ENSO Transition Probability Experiments - Models** 

Output CSV: 

- /home/563/ft3359/FT-Honours/Honours_Paper/Transition_Prob/TP_Models.csv

In [1]:
import os
import hashlib
import xarray as xr
import cftime
import numpy as np
import pandas as pd
 
 
# INPUTS
model_paths = {
    "ACCESS-ESM1.5":      "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_ACCESS-ESM1.5_r1i1p1_0850_1849_r240x120.nc",
    "BCC-CSM1-1":         "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_bcc-csm1-1_r1i1p1_0850_2000_r240x120.nc",
    "CCSM4":              "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_CCSM4_r1i1p1_0850_1850_r240x120.nc",
    "CSIRO-Mk3L-1-2":     "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_CSIRO-Mk3L-1-2_r1i1p1_0851_1850_r240x120.nc",
    "GISS-E2-R":          "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_GISS-E2-R_r1i1p121_0850_1850_r240x120.nc",
    "IPSL-CM5A-LR":       "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_IPSL-CM5A-LR_r1i1p1_0850_1850_r240x120.nc",
    "MIROC-ES2L":         "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_MIROC-ES2L_r1i1p1_0850_1849_r240x120.nc",
    "MIROC-ESM":          "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_MIROC-ESM_r1i1p1_0850_1849_r240x120.nc",
    "MPI-ESM-P":          "/g/data/ob22/jxb548/PMIPDATA/past1000/ts_MPI-ESM-P_r1i1p1_0850_1849_r240x120.nc",
}
 
# ERUPTION LIST: all 20 eruptions (850-1849 CE) with known seasonality.

ERUPTIONS_RAW = [
    (939.0,  4.0,  1.0,  63.6, -1.0, 16.23, 4.97),
    (946.0, 11.0,  1.0,  42.0, -1.0,  1.72, 0.61),
    (1257.0, 7.0,  1.0,  -8.4,  1.4, 59.42, 10.86),
    (1477.0, 2.0,  1.0,  64.6, -1.0,  5.12, 1.61),
    (1510.0, 7.0, 25.0,  64.0, -1.0,  2.30, 0.83),
    (1585.0, 1.0, 10.0,  19.5, 10.6,  8.51, 2.34),
    (1595.0, 3.0,  1.0,   4.9,  0.8,  8.87, 1.51),
    (1600.0, 2.0, 17.0, -16.6,  2.0, 18.95, 4.03),
    (1640.0, 12.0, 26.0,  6.1,  2.8, 18.68, 4.28),
    (1667.0, 9.0, 23.0,  42.7, -1.0,  3.48, 1.11),
    (1673.0, 5.0, 20.0,   1.4,  0.7,  4.67, 0.82),
    (1707.0, 12.0, 16.0,  35.4, -1.0,  1.08, 0.40),
    (1721.0, 5.0, 11.0,  63.6, -1.0,  0.81, 0.36),
    (1739.0, 8.0, 19.0,  42.7, -1.0,  3.44, 1.09),
    (1755.0, 10.0, 17.0,  63.6, -1.0,  1.18, 0.43),
    (1766.0, 4.0,  5.0,  64.0, -1.0,  2.52, 0.75),
    (1783.0, 6.0, 15.0,  64.4, -1.0, 20.81, 7.04),
    (1815.0, 4.0, 10.0,  -8.0,  0.8, 28.08, 4.49),
    (1822.0, 10.0,  8.0,  -7.3, 10.0,  2.02, 0.79),
    (1835.0, 1.0, 20.0,  13.0,  2.0,  9.48, 2.21),
]
eruptions_df = pd.DataFrame(
    ERUPTIONS_RAW,
    columns=["yearCE", "month", "day", "lat", "hemi", "ssi", "sigma_ssi"],
)
 
eruption_years = eruptions_df["yearCE"].to_numpy(dtype=int)

In [2]:
# SETTINGS
 
THRESHOLD = 0.5
EXCLUDE_MARGIN_YEARS = 5
PHASES = ["El Niño", "Neutral", "La Niña"]
CODE_TO_IDX = {-1: 0, 0: 1, 1: 2}
PHASE_TO_IDX_INTERNAL = {"La Niña": 0, "Neutral": 1, "El Niño": 2}
MIN_TRANSITIONS_FOR_PROBS = 1
 
# OUTPUT
WLP_CSV_OUT = "/home/563/ft3359/FT-Honours/Honours_Paper/Transition_Prob/All_Eruptions/TransProb_Models.csv"
 
 
# HELPERS
def load_sst_data(file_path: str) -> xr.DataArray:
    ds = xr.open_dataset(file_path, use_cftime=True)
    sst = ds["ts"]
    if float(sst.isel(time=0).max()) > 200:
        sst = sst - 273.15
    t0 = sst["time"].values[0]
    start_str = "0850-01-01"
    end_str   = "1849-12-31"
    if isinstance(t0, cftime.Datetime360Day):
        end_str = "1849-12-30"
    out = sst.sel(time=slice(start_str, end_str))
    ds.close()
    return out
 
 
def area_weighted_mean_latlon(da: xr.DataArray) -> xr.DataArray:
    w = np.cos(np.deg2rad(da["lat"]))
    w = xr.DataArray(w, coords={"lat": da["lat"]}, dims=("lat",))
    return da.weighted(w).mean(dim=["lat", "lon"])
 
 
def calculate_relative_nino34_monthly(sst_1000: xr.DataArray) -> xr.DataArray:
    n34  = sst_1000.sel(lat=slice(-5, 5), lon=slice(190, 240)).mean(dim=["lat", "lon"])
    trop = area_weighted_mean_latlon(sst_1000.sel(lat=slice(-20, 20)))
    n34_anom  = n34.groupby("time.month")  - n34.groupby("time.month").mean("time")
    trop_anom = trop.groupby("time.month") - trop.groupby("time.month").mean("time")
    rel = n34_anom - trop_anom
    return rel.rolling(time=3, center=True).mean()
 
 
def calculate_juljun_annual(monthly_3mo: xr.DataArray) -> xr.DataArray:
    t = monthly_3mo["time"]
    enso_year = xr.where(t.dt.month >= 7, t.dt.year, t.dt.year - 1)
    annual = monthly_3mo.groupby(enso_year).mean("time")
    years  = annual["group"].values.astype(int)
    annual = annual.rename({"group": "time"}).assign_coords(time=years)
    return annual
 
 
def classify_codes_vector(values: np.ndarray, thr: float) -> np.ndarray:
    out = np.full(values.shape, 999, dtype=np.int16)
    good = np.isfinite(values)
    v = values[good]
    out_good = np.zeros(v.shape, dtype=np.int16)
    out_good[v >= thr]  =  1
    out_good[v <= -thr] = -1
    out[good] = out_good
    return out
 
 
def in_eruption_margin_fast(years: np.ndarray, eruption_years: np.ndarray, margin: int) -> np.ndarray:
    years    = years.astype(int)
    e_sorted = np.sort(eruption_years.astype(int))
    lo = np.searchsorted(e_sorted, years - margin, side="left")
    hi = np.searchsorted(e_sorted, years + margin, side="right")
    return (hi > lo)
 
 
def sanitize_prob_vector(p: np.ndarray) -> np.ndarray:
    p = np.asarray(p, dtype=float).copy()
    p[~np.isfinite(p)] = 0.0
    p[p < 0] = 0.0
    s = p.sum()
    if s > 0:
        p = p / s
    else:
        p[:] = 0.0
    return p
 
 
def _internal_counts_to_plot_probs(counts_internal_LN_N_EN: np.ndarray) -> np.ndarray:

    p = counts_internal_LN_N_EN.astype(float)
    s = p.sum()
    if s <= 0:
        return np.zeros(3, dtype=float)
    p = p / s
    return np.array([p[2], p[1], p[0]], dtype=float)
 
 
def wlp_pool_probs(counts_by_item_base, counts_by_item_erupt,
                   phases=PHASES, nmin_each=MIN_TRANSITIONS_FOR_PROBS):
    pooled_probs_base  = {}
    pooled_probs_erupt = {}
    n_items_used = {}
    items_used   = {}
 
    for sp in phases:
        pB_list, pE_list, used = [], [], []
 
        for item in counts_by_item_base:
            cb = np.asarray(counts_by_item_base[item][sp],  dtype=float)
            ce = np.asarray(counts_by_item_erupt[item][sp], dtype=float)
 
            if int(cb.sum()) >= nmin_each and int(ce.sum()) >= nmin_each:
                pB = sanitize_prob_vector(_internal_counts_to_plot_probs(cb))
                pE = sanitize_prob_vector(_internal_counts_to_plot_probs(ce))
                if pB.sum() == 0 or pE.sum() == 0:
                    continue
                pB_list.append(pB)
                pE_list.append(pE)
                used.append(item)
 
        if not used:
            pooled_probs_base[sp]  = np.zeros(3, dtype=float)
            pooled_probs_erupt[sp] = np.zeros(3, dtype=float)
            n_items_used[sp] = 0
            items_used[sp]   = []
        else:
            pooled_probs_base[sp]  = sanitize_prob_vector(np.mean(np.vstack(pB_list), axis=0))
            pooled_probs_erupt[sp] = sanitize_prob_vector(np.mean(np.vstack(pE_list), axis=0))
            n_items_used[sp] = len(used)
            items_used[sp]   = used
 
    return pooled_probs_base, pooled_probs_erupt, n_items_used, items_used
 
 
def save_wlp_probs_csv(wlp_base, wlp_erupt, n_used, used_items, out_csv):
    rows = []
    for start_ph in PHASES:
        pb = np.asarray(wlp_base.get(start_ph,  [0, 0, 0]), dtype=float)
        pe = np.asarray(wlp_erupt.get(start_ph, [0, 0, 0]), dtype=float)
        rows.append({
            "start_phase":   start_ph,
            "baseline_EN":   float(pb[0]),
            "baseline_N":    float(pb[1]),
            "baseline_LN":   float(pb[2]),
            "eruption_EN":   float(pe[0]),
            "eruption_N":    float(pe[1]),
            "eruption_LN":   float(pe[2]),
            "n_items_used":  int(n_used.get(start_ph, 0)),
            "items_used":    "; ".join(used_items.get(start_ph, [])),
        })
    os.makedirs(os.path.dirname(out_csv), exist_ok=True)
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print("Saved WLP CSV:", out_csv)

In [3]:
# MAIN LOOP
counts_by_model_erupt = {}
counts_by_model_base  = {}
 
for model_name, path in model_paths.items():
    print(model_name)
 
    if not os.path.exists(path):
        print(f"  -> Missing file: {path} (skipping)")
        continue
 
    sst = load_sst_data(path)
    rel_monthly = calculate_relative_nino34_monthly(sst)
    rel_annual  = calculate_juljun_annual(rel_monthly).compute()
 
    years  = rel_annual["time"].values.astype(int)
    values = rel_annual.values.astype(float)
    codes  = classify_codes_vector(values, THRESHOLD)
 
    year_to_pos = {int(y): i for i, y in enumerate(years)}
    cand = np.array([y for y in years if (y + 1) in year_to_pos], dtype=int)
    if cand.size == 0:
        continue
 
    pos_y  = np.fromiter((year_to_pos[y]     for y in cand), count=cand.size, dtype=int)
    pos_y1 = np.fromiter((year_to_pos[y + 1] for y in cand), count=cand.size, dtype=int)
 
    mask_valid = (codes[pos_y] != 999) & (codes[pos_y1] != 999)
    cand   = cand[mask_valid]
    pos_y  = pos_y[mask_valid]
    pos_y1 = pos_y1[mask_valid]
    if cand.size == 0:
        continue
 
    start_codes = codes[pos_y].astype(int)
    next_codes  = codes[pos_y1].astype(int)
    start_idx   = np.array([CODE_TO_IDX[c] for c in start_codes], dtype=np.int8)
    next_idx    = np.array([CODE_TO_IDX[c] for c in next_codes],  dtype=np.int8)
 
    is_erupt_year    = np.isin(cand, eruption_years)
    in_margin        = in_eruption_margin_fast(cand, eruption_years, EXCLUDE_MARGIN_YEARS)
    is_baseline_year = ~in_margin
 
    model_counts_erupt = {ph: np.zeros(3, dtype=int) for ph in PHASES}
    model_counts_base  = {ph: np.zeros(3, dtype=int) for ph in PHASES}
 
    for ph in PHASES:
        ph_idx = PHASE_TO_IDX_INTERNAL[ph]
        erupt_mask = is_erupt_year    & (start_idx == ph_idx)
        base_mask  = is_baseline_year & (start_idx == ph_idx)
        model_counts_erupt[ph] += np.bincount(next_idx[erupt_mask], minlength=3).astype(int)
        model_counts_base[ph]  += np.bincount(next_idx[base_mask],  minlength=3).astype(int)
 
    counts_by_model_erupt[model_name] = model_counts_erupt
    counts_by_model_base[model_name]  = model_counts_base
 
 
# WLP POOLING AND CSV SAVE
wlp_probs_base, wlp_probs_erupt, wlp_n_used, wlp_used = wlp_pool_probs(
    counts_by_model_base, counts_by_model_erupt
)
 
save_wlp_probs_csv(wlp_probs_base, wlp_probs_erupt, wlp_n_used, wlp_used, WLP_CSV_OUT)
 

ACCESS-ESM1.5


/g/data/xp65/admin/analysis3/sitecustomize.py:72: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  mod = _real_import(name, globals, locals, fromlist, level)
/jobfs/173835307.gadi-pbs/ipykernel_3560831/688637233.py:16: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.open_dataset(file_path, use_cftime=True)


BCC-CSM1-1


/jobfs/173835307.gadi-pbs/ipykernel_3560831/688637233.py:16: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.open_dataset(file_path, use_cftime=True)


CCSM4


...

MIROC-ESM


/jobfs/173835307.gadi-pbs/ipykernel_3560831/688637233.py:16: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.open_dataset(file_path, use_cftime=True)


MPI-ESM-P


/jobfs/173835307.gadi-pbs/ipykernel_3560831/688637233.py:16: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.open_dataset(file_path, use_cftime=True)


Saved WLP CSV: /home/563/ft3359/FT-Honours/Honours_Paper/Transition_Prob/All_Eruptions/TransProb_Models.csv
